In [8]:
import pandas as pd
import numpy as np

data_path = (
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/flavor_globalfit/"
    "hese/combined_with_bdt/data_HESE_pass2/"
    "mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/"
    "dataset_data_HESE_pass2.parquet"
)

df = pd.read_parquet(data_path)
print(f"Loaded {len(df):,} events")

df = df[df["reco_energy"] > 60e3]  # 60 TeV in GeV
print(f"After reco_energy > 60 TeV: {len(df):,} events")

Loaded 193 events
After reco_energy > 60 TeV: 115 events


In [9]:
features = [
    'TauMonoDiff_rlogl',
    'Taupede_Asymmetry',
    'Taupede_Distance',
    'Taupede1_Particles_energy',
    'Taupede2_Particles_energy',
    'cscdSBU_MonopodFit4_noDC_zenith',
    'MonopodFit_iMIGRAD_PPB0_Delay_ice',
    'CVStatistics_q_max_doms',
    'cscdSBU_VertexRecoDist_CscdLLh',
    'MonopodFit_iMIGRAD_PPB0_energy',
    'cscdSBU_Qtot_HLC_log',
    'TauMonoMilliDiff_rlogl',
    'TauSPEMilliDiff_rlogl',
    'econfinement',
    'EventGeneratorDC_Thijs_length',
    'RecoERatio_EventGeneratorDC_Max',
]

print(f"{'Feature':<45} {'Min':>15} {'Max':>15}")
print("-" * 77)
for feat in features:
    if feat in df.columns:
        vmin = df[feat].min()
        vmax = df[feat].max()
        print(f"{feat:<45} {vmin:>15.4f} {vmax:>15.4f}")
    else:
        print(f"{feat:<45} {'NOT FOUND':>15}")

Feature                                                   Min             Max
-----------------------------------------------------------------------------
TauMonoDiff_rlogl                                     -8.4185          0.0040
Taupede_Asymmetry                                     -0.9958          1.0000
Taupede_Distance                                       1.8485        763.0585
Taupede1_Particles_energy                            234.3036 10349468077.6835
Taupede2_Particles_energy                              0.0000   48088504.1890
cscdSBU_MonopodFit4_noDC_zenith                        0.1730          2.9987
MonopodFit_iMIGRAD_PPB0_Delay_ice                 -14278.6785        387.0905
CVStatistics_q_max_doms                              338.6650      21365.9350
cscdSBU_VertexRecoDist_CscdLLh                         3.6682        440.3704
MonopodFit_iMIGRAD_PPB0_energy                     44401.1517  119850944.2076
cscdSBU_Qtot_HLC_log                                   3.8158  

In [10]:
import os

N_EDGES = 11  # 10 bins
OUT_DIR = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/override/binning/hese/combined"

pairs = [
    ('Taupede_Distance',               'Taupede_Asymmetry'),
    ('Taupede1_Particles_energy',      'Taupede2_Particles_energy'),
    ('MonopodFit_iMIGRAD_PPB0_energy', 'cscdSBU_MonopodFit4_noDC_zenith'),
    ('MonopodFit_iMIGRAD_PPB0_Delay_ice', 'CVStatistics_q_max_doms'),
    ('cscdSBU_VertexRecoDist_CscdLLh', 'cscdSBU_Qtot_HLC_log'),
    ('TauMonoDiff_rlogl',              'econfinement'),
    ('TauSPEMilliDiff_rlogl',          'TauMonoMilliDiff_rlogl'),
    ('EventGeneratorDC_Thijs_length',  'RecoERatio_EventGeneratorDC_Max'),
]

log_vars = {
    'Taupede_Distance',
    'Taupede1_Particles_energy',
    'Taupede2_Particles_energy',
    'MonopodFit_iMIGRAD_PPB0_energy',
}

fixed_ranges = {
    'Taupede_Asymmetry': (-1.0, 1.0),
    'econfinement':      (0.0,  1.0),
}

def padded_range(vmin, vmax, pad_frac=0.05):
    pad = pad_frac * (vmax - vmin)
    return vmin - pad, vmax + pad

def var_binning(feat):
    vals = df[feat]
    if feat in fixed_ranges:
        b_min, b_max = fixed_ranges[feat]
        return f"{feat}_binning = ({b_min:.4f}, {b_max:.4f}, {N_EDGES}, lin)"
    elif feat in log_vars:
        log_min = np.log10(vals[vals > 0].min())
        log_max = np.log10(vals.max())
        b_min, b_max = padded_range(log_min, log_max)
        return f"{feat}_binning = ({b_min:.4f}, {b_max:.4f}, {N_EDGES}, log)"
    else:
        b_min, b_max = padded_range(vals.min(), vals.max())
        return f"{feat}_binning = ({b_min:.4f}, {b_max:.4f}, {N_EDGES}, lin)"

def make_config(var1, var2):
    lines = []
    lines.append(f"[IC86_pass2_SnowStorm_FTP_HESE_Combined_2D_binning]")
    lines.append(f"class_name = Binning_2D")
    lines.append(f"analysis_variables = {var1},{var2}")
    for feat in (var1, var2):
        lines.append(var_binning(feat))
    lines.append("")
    lines.append(f"[IC86_pass2_SnowStorm_FTP_HESE_Combined_var_mapping]")
    for feat in (var1, var2):
        lines.append(f"{feat} = {feat}")
    lines.append("")
    return "\n".join(lines)

for v1, v2 in pairs:
    missing = [f for f in (v1, v2) if f not in df.columns]
    if missing:
        print(f"SKIPPED ({', '.join(missing)} not in data): {v1} x {v2}")
        continue

    content = make_config(v1, v2)
    fname = os.path.join(OUT_DIR, f"{v1}_{v2}_10bins.cfg")
    with open(fname, "w") as fh:
        fh.write(content)
    print(f"Written: {fname}")
    print(content)
    print("-" * 60)

Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/override/binning/hese/combined/Taupede_Distance_Taupede_Asymmetry_10bins.cfg
[IC86_pass2_SnowStorm_FTP_HESE_Combined_2D_binning]
class_name = Binning_2D
analysis_variables = Taupede_Distance,Taupede_Asymmetry
Taupede_Distance_binning = (0.1360, 3.0133, 11, log)
Taupede_Asymmetry_binning = (-1.0000, 1.0000, 11, lin)

[IC86_pass2_SnowStorm_FTP_HESE_Combined_var_mapping]
Taupede_Distance = Taupede_Distance
Taupede_Asymmetry = Taupede_Asymmetry

------------------------------------------------------------
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/override/binning/hese/combined/Taupede1_Particles_energy_Taupede2_Particles_energy_10bins.cfg
[IC86_pass2_SnowStorm_FTP_HESE_Combined_2D_binning]
class_name = Binning_2D
analysis_variables = Taupede1_Particles_energy,Taupede2_Particles_energy
Taupede1_Particles_energy_binning = (1.9875, 10.3972, 11, log)
Taupede2_Part